# 2.8b — `vit_style`, our own transformer

**One arm, one notebook.** Split out of `28_capacity_and_resolution.ipynb` so it can run on
its own; `00` and `01` there are finished and `02` was stopped.

This is **the last architecture of ours with no number at all.** Seven of our eight models
have been measured; `vit_style` is the eighth.

## What it isolates

Sized to **2.83M parameters** — the same as `v27-resnet_style` (0.8900) and within 6% of
`v28-convnext_big_128` (0.8988). So against those two it is *attention versus convolution
at matched capacity*, with nothing else moving.

patch 16 over a 128px map gives 64 tokens — the same sequence length `vit_style` would get
from patch 8 at 64px, with each token carrying 4x the pixels.

## The one deviation, stated up front

This arm runs **cosine warmup**; every convolutional arm runs at constant learning rate. A
from-scratch ViT with no warmup collapses in the first few hundred steps for reasons that
have nothing to do with attention, so the alternative was reporting a failure that meant
nothing. It is not schedule-matched to the CNNs, and that belongs in the presentation.

## What to expect

A transformer trained from scratch on 121k images, with no pretraining and no heavy
augmentation stack, usually loses to a CNN of the same size — transformers lack the
locality prior and buy it back with data we do not have. **If it loses, that is a clean
finding**, and it pairs with the pretrained results: `v30`/`v32` show attention and large
capacity only pay when someone else already spent the data.

## Before you start

1. `git pull` in `/content/fdl-project`, then **Runtime > Restart session** — Colab caches
   `fdl_project` after the first import, so a pull alone does nothing.
2. `wandb login`, or `WANDB_KEY` in Colab Secrets.
3. Run All.

## 0. Colab web UI only — clone and authenticate

Skip if `/content/fdl-project` already exists.

In [1]:
from getpass import getpass
from pathlib import Path
import subprocess

TARGET = Path("/content/fdl-project")
BRANCH = "feature/phase3-architectures"
REMOTE = "github.com/ezero3/fdl-project.git"


def run(*command: str) -> None:
    subprocess.run(command, check=True)


if TARGET.exists():
    print(f"{TARGET} already present -- pulling")
    run("git", "-C", str(TARGET), "fetch", "origin", BRANCH)
    run("git", "-C", str(TARGET), "checkout", BRANCH)
    run("git", "-C", str(TARGET), "pull", "--ff-only")
else:
    # Private repo, so the clone needs a personal access token. getpass keeps it
    # out of the notebook and out of the output.
    token = getpass("GitHub personal access token (input hidden): ").strip()
    run("git", "clone", "--branch", BRANCH,
        f"https://{token}@{REMOTE}", str(TARGET))
    # Drop the token from the stored remote; a later pull will ask again rather
    # than leaving a credential sitting in .git/config.
    run("git", "-C", str(TARGET), "remote", "set-url", "origin", f"https://{REMOTE}")
    del token

print(subprocess.run(["git", "-C", str(TARGET), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

/content/fdl-project already present -- pulling
590e5ab feat: 🎯 split vit_style into its own notebook


## 1. Setup

Mounts Drive, copies the dataset, installs the package.

In [2]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

Mounted at /content/drive
gpu     NVIDIA A100-SXM4-80GB
drive   mounted
dataset 1.88 GiB


## 2. W&B

In [3]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        # Colab Secrets work on the web UI. They time out under the VS Code
        # runtime, which is why the other notebooks tell you to use a terminal.
        # Add WANDB_KEY at the key icon in the left sidebar and enable it here.
        try:
            from google.colab import userdata

            wandb.login(key=userdata.get("WANDB_KEY"))
        except Exception as error:
            print(f"  Colab Secrets unavailable ({type(error).__name__})")

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- add WANDB_KEY to Colab Secrets, or run "
              "`wandb login` in a terminal, then rerun this cell")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")


  wandb ready, project 'wm811k-wafer-defects'


## 3. Train

One config: `configs/train/v28_capacity/03_vit_style_128.yaml`.

Checkpoints go to Drive under this arm's own name, so if the runtime drops, rerunning this
cell resumes from the last epoch rather than restarting.

In [4]:
import shutil, time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.models.baseline_cnn import count_trainable_parameters
from fdl_project.training.runner import run_experiment

SERIES = "v28_capacity"
CONFIG = REPO / "configs/train" / SERIES / "03_vit_style_128.yaml"
assert CONFIG.exists(), f"{CONFIG} missing -- git pull, then Runtime > Restart session"

OUTPUT = REPO / "output" / SERIES
OUTPUT.mkdir(parents=True, exist_ok=True)

OVERRIDES = ["data.transform_device=cuda"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}",
                  f"logging.wandb.tags=[{SERIES},from_scratch]"]

config = load_experiment_config(CONFIG, overrides=OVERRIDES)
assert config.model.name == "vit_style", "this notebook runs one arm: vit_style"
assert config.data.augmentation.name == "rotation", "settled pipeline is rotation"

model = build_model(config.model.name, **config.model.kwargs)
parameters = count_trainable_parameters(model)
del model
print(f"=== {config.name}")
print(f"    {parameters:,} parameters, {config.data.preprocessing.target_size[0]}px, "
      f"patch {config.model.kwargs['patch_size']}, depth {config.model.kwargs['depth']}")
print(f"    schedule {config.scheduler.name}, max_epochs {config.trainer.max_epochs}, "
      f"patience {config.trainer.early_stopping.patience}")

dataframe = load_wm811k_dataframe(DATASET)
started = time.monotonic()
result = run_experiment(config, overwrite=True, dataframe=dataframe)

macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
per_class = result.validation.per_class_metrics.set_index("class_name")["f1"]
row = {
    "run": config.name,
    "model": config.model.name,
    "params": parameters,
    "px": config.data.preprocessing.target_size[0],
    "macro_f1": round(float(macro.point_estimate), 4),
    "ci_lower": round(float(macro.ci_lower), 4),
    "ci_upper": round(float(macro.ci_upper), 4),
    "scratch_f1": round(float(per_class["Scratch"]), 3),
    "near_full_f1": round(float(per_class["Near-full"]), 3),
    "best_epoch": result.fit.best_epoch,
    "epochs": len(result.fit.history),
    "minutes": round((time.monotonic() - started) / 60, 1),
}
csv = OUTPUT / "vit_style_128_results.csv"
pd.DataFrame([row]).to_csv(csv, index=False)
if HAS_DRIVE:
    shutil.copy2(csv, DRIVE / f"{SERIES}_vit_style_128.csv")

print(f"\n  macro-F1 {row['macro_f1']:.4f} [{row['ci_lower']:.4f}, {row['ci_upper']:.4f}]")
print(f"  Scratch {row['scratch_f1']:.3f}   Near-full {row['near_full_f1']:.3f}")
print(f"  best epoch {row['best_epoch']}/{row['epochs']}   {row['minutes']:.1f} min")
if row["best_epoch"] >= row["epochs"] - 2:
    print("  STILL IMPROVING AT THE CAP -- this number is a floor.")
print(f"  saved to {csv}")

=== v28-vit_style_128
    2,831,625 parameters, 128px, patch 16, depth 6
    schedule cosine_with_warmup, max_epochs 100, patience 10


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: vlad-yelisieiev-bicocca (vlad-yelisieiev-bicocca-milano-bicocca) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇████
epoch_seconds,▅▆▆▆█▄▂▄▄▅▁▂▅▃▂▄▅▃▂▄▄▃▁▄▅▅▃▃▂▇▄▂▆▃▄▂▄▂▃▁
learning_rate,▂█████████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▄▄▃▃▃▃▃▃▃▂▂▂▂▁▁▁
train_accuracy,▁▂▃▃▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇██████████
train_loss,█▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_samples,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation_accuracy,▂▁▅▄▄▆▆▄▆▆▂▆▇▇▇▇▆▇▇▅▇▇▇▇▇▇▇▇█▇▇▇▇█▇█████
validation_balanced_accuracy,▁▃▄▄▂▅▆▆█▇█▇▆▇▇▇█▇█▇▇▇██▇▇▇▇███▇███▇▇█▇█
validation_f1_Center,▁▁▇▆▇▇▆▇▅▆▇▇▇▇█▇▇▇▇█▆▇█▇▇█▇▇▇▇█▇██▇▇████
validation_f1_Donut,▁▅▆▇▅▇▅▇███▇██▇█▇▇███▇▇▇███▇▇███▇█▇█████
+10,...



  macro-F1 0.8744 [0.8615, 0.8844]
  Scratch 0.668   Near-full 0.966
  best epoch 70/80   19.1 min
  saved to /content/fdl-project/output/v28_capacity/vit_style_128_results.csv


## 4. Where it lands

Run after section 3 finishes.

In [5]:
# All measured on the same splits, augmentation and sampler.
REFERENCE = [
    ("v32-resnet34_finetune",      "pretrained, 21.4M",   0.9041, 0.793),
    ("dilated-style-64-dihedral8", "ours, 298k, old aug", 0.8995, 0.793),
    ("v28-convnext_big_128",       "ours, 2.68M, 128px",  0.8988, 0.787),
    ("v30-finetune_encoder_1e-5",  "pretrained, 11.3M",   0.8938, 0.813),
    ("v27-resnet_style",           "ours, 2.83M, 64px",   0.8900, 0.759),
    ("v27-convnext_style",         "ours, 414k, 64px",    0.8883, 0.736),
    ("v28-convnext_big_64",        "ours, 2.68M, 64px",   0.8862, 0.740),
    ("v27-baseline_cnn",           "ours, 157k, 64px",    0.8646, 0.715),
]
NOISE_FLOOR = 0.02

table = pd.DataFrame(REFERENCE, columns=["run", "what", "macro_f1", "scratch_f1"])
mine = pd.DataFrame([{"run": row["run"], "what": "THIS NOTEBOOK (ours, 2.83M)",
                      "macro_f1": row["macro_f1"], "scratch_f1": row["scratch_f1"]}])
pd.set_option("display.width", 210)
display(pd.concat([mine, table]).sort_values("macro_f1", ascending=False).reset_index(drop=True))

# The comparison this arm exists for: same capacity, convolution vs attention.
for label, reference in [("v27-resnet_style (2.83M, conv, 64px)", 0.8900),
                         ("v28-convnext_big_128 (2.68M, conv, 128px)", 0.8988)]:
    delta = row["macro_f1"] - reference
    verdict = "REAL" if abs(delta) > NOISE_FLOOR else "inside the noise floor"
    print(f"vs {label:44} {delta:+.4f}  [{verdict}]")

,run,what,macro_f1,scratch_f1
0,v32-resnet34_finetune,"pretrained, 21.4M",0.9041,0.793
1,dilated-style-64-dihedral8,"ours, 298k, old aug",0.8995,0.793
2,v28-convnext_big_128,"ours, 2.68M, 128px",0.8988,0.787
3,v30-finetune_encoder_1e-5,"pretrained, 11.3M",0.8938,0.813
4,v27-resnet_style,"ours, 2.83M, 64px",0.8900,0.759
5,v27-convnext_style,"ours, 414k, 64px",0.8883,0.736
6,v28-convnext_big_64,"ours, 2.68M, 64px",0.8862,0.740
7,v28-vit_style_128,"THIS NOTEBOOK (ours, 2.83M)",0.8744,0.668
8,v27-baseline_cnn,"ours, 157k, 64px",0.8646,0.715


vs v27-resnet_style (2.83M, conv, 64px)         -0.0156  [inside the noise floor]
vs v28-convnext_big_128 (2.68M, conv, 128px)    -0.0244  [REAL]


## 5. Report back

```
vit_style_128: macro-F1 X.XXXX [lo, hi], Scratch X.XXX, best epoch N/M, T min
```

Flag it if the run says **STILL IMPROVING AT THE CAP** — the cap is 100 epochs here, but a
from-scratch transformer converges slowly and could still hit it.

CSV at `output/v28_capacity/vit_style_128_results.csv`, copied to Drive automatically.